# L8: Financial and temporal features

Derive features with documented formulas and prediction-time availability.

**Environment:** See ../requirements.txt and the labs index. Examples use bundled inputs; no internet is required after dependency installation. Synthetic data is for teaching only.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd() if (Path.cwd() / "datasets").exists() else Path.cwd().parent
assert (ROOT / "datasets").exists(), "Run from the repository root or labs directory; in Colab upload/clone the repository first."
DATA = ROOT / "datasets"


## Transaction features

In [ ]:
sale=pd.DataFrame({"Quantity":[3],"UnitPrice":[200],"Discount":[.10],"UnitCost":[120],"OrderDate":["2024-01-08"]})
sale["GrossRevenue"]=sale.Quantity*sale.UnitPrice
sale["NetRevenue"]=sale.GrossRevenue*(1-sale.Discount)
sale["TotalCost"]=sale.Quantity*sale.UnitCost
sale["Profit"]=sale.NetRevenue-sale.TotalCost
sale["ProfitMargin"]=sale.Profit/sale.NetRevenue
sale["OrderDate"]=pd.to_datetime(sale.OrderDate)
sale["Month"]=sale.OrderDate.dt.month
sale["Weekday"]=sale.OrderDate.dt.day_name()
assert sale.Profit.iloc[0]==180
print(sale.T)

## Past-only customer feature

In [ ]:
df=pd.read_csv(DATA / "sales_multiyear.csv",parse_dates=["OrderDate"])
daily=df.groupby(["CustomerKey","OrderDate"],as_index=False).NetRevenue.sum().sort_values(["CustomerKey","OrderDate"])
daily["PreviousCustomerDayRevenue"]=daily.groupby("CustomerKey").NetRevenue.shift(1)
daily["PreviousObservedDate"]=daily.groupby("CustomerKey").OrderDate.shift(1)
assert (daily.PreviousObservedDate.dropna() < daily.loc[daily.PreviousObservedDate.notna(),"OrderDate"]).all()
print(daily.head())
# Previous observed customer day, not necessarily yesterday; excludes same-day information.


## Student task
Give one feature unavailable when predicting an order before checkout. Create price bands with explicit inclusive boundaries.

## Reference solution
Try the task before reading this cell.

In [ ]:
df["PriceBand"]=pd.cut(df.UnitPrice,[0,25,200,np.inf],labels=["Budget","Mid","Premium"],include_lowest=True)
print(df.PriceBand.value_counts())

## Interpretation and limitations
Record the dataset grain, units, assumptions, and a limitation in your submission. Explain one result rather than only submitting screenshots.